# ViGovBot — PDF to TTHC chunks on Google Colab

In [ ]:
%pip install -q pymupdf4llm==1.28.2 python-dotenv==1.2.3 'pydantic>=2.10,<3' pyyaml==6.0.3

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/qtamtensor05/ViGovBot.git"
PROJECT_DIR = Path("/content/ViGovBot")

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f"Project: {PROJECT_DIR}")

In [ ]:
import os
from google.colab import files

from src.configuration import load_configuration
CONFIG_PATH = PROJECT_DIR / 'config.yaml'
pipeline_settings, ingestion_settings, chunking_settings = load_configuration(CONFIG_PATH)
USE_GOOGLE_DRIVE = pipeline_settings.colab.use_google_drive
DRIVE_INPUT_DIR = pipeline_settings.colab.drive_input
OUTPUT_DIR = Path(pipeline_settings.colab.output)
OVERWRITE = pipeline_settings.overwrite

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    input_root = Path(DRIVE_INPUT_DIR)
    if not input_root.is_dir():
        raise FileNotFoundError(f"Không tìm thấy thư mục: {input_root}")
    pdf_files = sorted(path for path in input_root.rglob("*") if path.is_file() and path.suffix.lower() == ".pdf")
else:
    uploaded = files.upload()
    pdf_files = sorted(Path("/content") / name for name in uploaded if Path(name).suffix.lower() == ".pdf")

if not pdf_files:
    raise ValueError("Không có file PDF nào được chọn.")
print(f"Đã chọn {len(pdf_files)} PDF")

In [ ]:
import time
# Reload after git pull: Python otherwise keeps the previously imported implementation.
import importlib
import src.chunking.markdown as chunking_module
import src.ingestion.parse as ingestion_module
importlib.invalidate_caches()
importlib.reload(chunking_module)
importlib.reload(ingestion_module)
parse_pdf_to_hybrid_data = ingestion_module.parse_pdf_to_hybrid_data
assert chunking_module.sanitize_text('bằng<br />đường ống') == 'bằng đường ống', 'Cần cập nhật mã nguồn GitHub'
assert chunking_module.TTHCStructureAwareChunker().classify_section('CƠ QUAN THỰC HIỆN') == 'metadata_identity'
print('Loaded source:', chunking_module.__file__)
subprocess.run(['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], check=True)

results = []
failures = []
started = time.perf_counter()

for index, pdf_path in enumerate(pdf_files, start=1):
    relative_parent = pdf_path.relative_to(input_root).parent if USE_GOOGLE_DRIVE else Path()
    target_dir = OUTPUT_DIR / relative_parent
    target_path = target_dir / f"{pdf_path.stem}.json"
    if target_path.exists() and not OVERWRITE:
        print(f"[{index}/{len(pdf_files)}] Bỏ qua: {pdf_path.name}")
        continue
    try:
        output_path, chunk_count = parse_pdf_to_hybrid_data(pdf_path, output_dir=target_dir, config_path=CONFIG_PATH)
        results.append({"source": str(pdf_path), "output": str(output_path), "chunks": chunk_count})
        print(f"[{index}/{len(pdf_files)}] OK: {pdf_path.name} → {chunk_count} chunks")
    except Exception as error:
        failures.append({"source": str(pdf_path), "error": f"{type(error).__name__}: {error}"})
        print(f"[{index}/{len(pdf_files)}] Lỗi: {pdf_path.name} — {error}")

elapsed = time.perf_counter() - started
print(f"Hoàn tất {len(results)}/{len(pdf_files)} file trong {elapsed:.2f}s; lỗi: {len(failures)}")

In [ ]:
import json
from pprint import pprint

pprint(results)
if failures:
    print("\nCác file lỗi:")
    pprint(failures)

if results:
    sample_chunks = json.loads(Path(results[0]["output"]).read_text(encoding="utf-8"))
    print("\nChunk mẫu:")
    print(json.dumps(sample_chunks[0], ensure_ascii=False, indent=2))

In [ ]:
import shutil

archive_path = shutil.make_archive("/content/ViGovBot_outputs", "zip", root_dir=OUTPUT_DIR)
print(f"Đã tạo: {archive_path}")
files.download(archive_path)